# OpenCV 動き検知ツール

固定カメラ映像でフレーム差分を使って「何か動いている区間」を検出する。

- 固定カメラなので背景は変わらない → 動いているピクセルだけが差分として現れる
- 動きがある区間の開始・終了時刻をCSVに出力
- 動きのある代表フレームをサムネイルとして保存（目視確認用）

In [ ]:
import cv2
import numpy as np
import pandas as pd
from pathlib import Path

# =====================
# パス設定
# =====================
VIDEO_PATH   = r'F:\ワオキツネザルCAM1\ここに動画のフルパスを貼る.mp4'
OUTPUT_DIR   = r'F:\thesis\motion_results'
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

# =====================
# パラメータ（調整可能）
# =====================
MOTION_THRESHOLD  = 500   # 動きと判定するピクセル数の閾値（小さくすると敏感になる）
MIN_DURATION_SEC  = 3     # 何秒以上動いていたら「動き区間」とするか
SAMPLE_INTERVAL   = 15    # 何フレームに1回チェックするか（15fps動画なら15=1秒に1回）
THUMBNAIL_INTERVAL= 30    # 何秒ごとにサムネイルを保存するか

print('✅ 設定完了')

## 動画情報を確認する

In [ ]:
cap = cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened():
    print('❌ 動画を開けませんでした。パスを確認してください。')
else:
    fps          = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    total_sec    = total_frames / fps
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()
    print(f'✅ 動画確認OK')
    print(f'   ファイル  : {Path(VIDEO_PATH).name}')
    print(f'   解像度    : {w}x{h}')
    print(f'   FPS       : {fps:.1f}')
    print(f'   総再生時間: {int(total_sec//60):02d}:{int(total_sec%60):02d} ({total_sec:.0f}秒)')

## 動き検知を実行する

In [ ]:
cap = cv2.VideoCapture(VIDEO_PATH)
stem = Path(VIDEO_PATH).stem
thumb_dir = Path(OUTPUT_DIR) / stem
thumb_dir.mkdir(exist_ok=True)

prev_gray    = None
frame_idx    = 0
motion_log   = []   # (time_sec, motion_pixels)
thumb_saved  = set()

while True:
    ret, frame = cap.read()
    if not ret:
        break

    if frame_idx % SAMPLE_INTERVAL == 0:
        time_sec = frame_idx / fps
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        gray = cv2.GaussianBlur(gray, (21, 21), 0)  # ノイズ除去

        if prev_gray is not None:
            diff        = cv2.absdiff(prev_gray, gray)
            _, thresh   = cv2.threshold(diff, 25, 255, cv2.THRESH_BINARY)
            motion_px   = int(np.sum(thresh > 0))
            is_motion   = motion_px > MOTION_THRESHOLD
            motion_log.append({
                'time_sec'  : time_sec,
                'time_str'  : f'{int(time_sec//60):02d}:{int(time_sec%60):02d}',
                'motion_px' : motion_px,
                'is_motion' : is_motion,
            })

            # 動きがある区間のサムネイルを保存
            thumb_key = int(time_sec // THUMBNAIL_INTERVAL)
            if is_motion and thumb_key not in thumb_saved:
                thumb_path = str(thumb_dir / f'{int(time_sec):05d}sec.jpg')
                cv2.imwrite(thumb_path, frame)
                thumb_saved.add(thumb_key)

        prev_gray = gray

    frame_idx += 1

cap.release()

motion_df = pd.DataFrame(motion_log)
print(f'✅ 解析完了')
print(f'   チェックしたポイント数: {len(motion_df)}')
print(f'   動きありポイント数    : {motion_df["is_motion"].sum()}')

## 動き区間をまとめてCSVに出力する

In [ ]:
# 連続する「動きあり」フレームをひとつの区間にまとめる
segments = []
in_motion   = False
seg_start   = None

for _, row in motion_df.iterrows():
    if row['is_motion'] and not in_motion:
        in_motion = True
        seg_start = row['time_sec']
    elif not row['is_motion'] and in_motion:
        in_motion = False
        seg_end   = row['time_sec']
        duration  = seg_end - seg_start
        if duration >= MIN_DURATION_SEC:
            segments.append({
                'start_sec' : seg_start,
                'end_sec'   : seg_end,
                'duration'  : duration,
                'start_str' : f'{int(seg_start//60):02d}:{int(seg_start%60):02d}',
                'end_str'   : f'{int(seg_end//60):02d}:{int(seg_end%60):02d}',
            })

# 最後まで動きが続いていた場合
if in_motion:
    seg_end  = motion_df['time_sec'].iloc[-1]
    duration = seg_end - seg_start
    if duration >= MIN_DURATION_SEC:
        segments.append({
            'start_sec': seg_start, 'end_sec': seg_end, 'duration': duration,
            'start_str': f'{int(seg_start//60):02d}:{int(seg_start%60):02d}',
            'end_str'  : f'{int(seg_end//60):02d}:{int(seg_end%60):02d}',
        })

seg_df = pd.DataFrame(segments)
csv_path = str(Path(OUTPUT_DIR) / f'{stem}_motion_segments.csv')
seg_df.to_csv(csv_path, index=False, encoding='utf-8-sig')

print(f'✅ 動き区間 {len(seg_df)} 件を検出')
print(f'   保存先: {csv_path}')
print()
print(seg_df[['start_str','end_str','duration']].to_string(index=False))

## サムネイル一覧を確認する（Jupyter上で表示）

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import glob

thumb_files = sorted(glob.glob(str(thumb_dir / '*.jpg')))

if not thumb_files:
    print('サムネイルが0件でした。MOTION_THRESHOLDを小さくしてみてください。')
else:
    cols = 4
    rows = (len(thumb_files) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(16, rows * 3))
    axes = axes.flatten() if rows > 1 else [axes] if cols == 1 else axes.flatten()

    for i, fpath in enumerate(thumb_files):
        img = mpimg.imread(fpath)
        axes[i].imshow(img)
        axes[i].set_title(Path(fpath).stem, fontsize=8)
        axes[i].axis('off')

    for j in range(i+1, len(axes)):
        axes[j].axis('off')

    plt.suptitle(f'動き検知サムネイル一覧：{stem}', fontsize=12)
    plt.tight_layout()
    plt.savefig(str(Path(OUTPUT_DIR) / f'{stem}_thumbnails.png'), dpi=100)
    plt.show()
    print(f'✅ サムネイル一覧を保存しました')